# Project Summary: SEND Energy Demand Response and EV Charging

This notebook gives a compact check of the full project.

The project asks how forecasted renewable surplus can be turned into EV charging actions while protecting driver service.

The workflow has six linked parts:

1. Prepare SEND campus energy data and Solcast weather data.
2. Calculate a potential renewable-surplus signal.
3. Forecast surplus from time, energy and weather features.
4. Build synthetic Keele EV scenarios from ACN workplace charging data.
5. Compare baseline, optimised and rolling-horizon charging strategies.
6. Explain forecasts and schedule changes with SHAP and counterfactual outputs.

In [1]:
from pathlib import Path
import os
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 120)


def find_project_root(start: Path | None = None) -> Path:
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "Data").exists() and (candidate / "Notebooks").exists():
            return candidate
    raise FileNotFoundError("Could not find the project root containing Data/ and Notebooks/.")


ROOT = find_project_root()
os.chdir(ROOT)

DATA_DIR = ROOT / "Data"
RESULTS_DIR = DATA_DIR / "Results"
PROCESSED_DIR = DATA_DIR / "Processed"
ACN_DIR = DATA_DIR / "ACN"
FIGURES_DIR = ROOT / "Figures"
MODELS_DIR = ROOT / "Models" / "Forecasting"
INTERVAL_MINUTES = 5
INTERVAL_HOURS = INTERVAL_MINUTES / 60

print("Project root:", ROOT)

Project root: /Users/mac/Send-Ev-Project/SEND-EV-Project


## Reproducible Workflow

Run the notebooks in this order when rebuilding the project from raw data:

1. `Format_DEOP.ipynb`
2. `Format_Solcast.ipynb`
3. `Solar_Missing.ipynb`
4. `Data_Loader.ipynb`
5. `Feature_Loader.ipynb`
6. `Training_Models.ipynb`
7. `Analysis_Solar.ipynb`
8. `Analysis_Consumption.ipynb`
9. `Renewable_Surplus.ipynb`
10. `ACN_Data_Acquisition.ipynb`
11. `ACN_Data_Cleaning_and_Flexibility.ipynb`
12. `EV_Scenario_Generation.ipynb`
13. `Charging_Baselines.ipynb`
14. `EV_Charging_Optimisation.ipynb`
15. `Rolling_Horizon_Control.ipynb`
16. `SHAP_Analysis.ipynb`
17. `Counterfactual_Explanations.ipynb`
18. `Final_Evaluation.ipynb`
19. `Project_Summary.ipynb`

Notebook ACN_Data_acquistion can reuse downloaded ACN files. A fresh API download needs `ACN_API_TOKEN`.

In [2]:
def read_table(path: Path, parse_dates: list[str] | None = None) -> pd.DataFrame:
    candidates = [path]
    if path.suffix.lower() == ".parquet":
        candidates.append(path.with_suffix(".csv"))
    elif path.suffix.lower() == ".csv":
        candidates.append(path.with_suffix(".parquet"))

    parquet_error: Exception | None = None

    for candidate in candidates:
        if candidate.exists():
            if candidate.suffix.lower() == ".parquet":
                try:
                    return pd.read_parquet(candidate)
                except (ImportError, ModuleNotFoundError) as exc:
                    parquet_error = exc
                    continue
            return pd.read_csv(candidate, parse_dates=parse_dates)

    if parquet_error is not None:
        raise ImportError(
            "A Parquet file exists, but pandas cannot read it and no CSV fallback was found. "
            "Install pyarrow or fastparquet, or rerun the previous notebook to create the CSV fallback."
        ) from parquet_error

    raise FileNotFoundError(f"None of these files exists: {candidates}")


expected_outputs = {
    "SEND repaired 2022": DATA_DIR / "DEOP" / "2022_DEOP_Repaired.csv",
    "SEND repaired 2023": DATA_DIR / "DEOP" / "2023_DEOP_Repaired.csv",
    "Solcast 2022": DATA_DIR / "Solcast" / "Solcast_2022.csv",
    "Solcast 2023": DATA_DIR / "Solcast" / "Solcast_2023.csv",
    "Surplus time series": PROCESSED_DIR / "send_solcast_surplus_5min.parquet",
    "Surplus events": RESULTS_DIR / "surplus_events.csv",
    "ACN acquisition manifest": ACN_DIR / "acquired" / "acquisition_manifest.csv",
    "ACN cleaned sessions": ACN_DIR / "processed" / "acn_sessions_cleaned.parquet",
    "ACN optimisation-ready sessions": ACN_DIR / "processed" / "acn_sessions_optimisation_ready.parquet",
    "Keele EV scenarios": ACN_DIR / "processed" / "keele_ev_scenarios.parquet",
    "Surplus predictions": RESULTS_DIR / "surplus_predictions.csv",
    "SHAP/contribution importance": RESULTS_DIR / "shap_feature_importance.csv",
    "Baseline metrics": RESULTS_DIR / "baseline_metrics.csv",
    "Optimisation metrics": RESULTS_DIR / "optimisation_metrics.csv",
    "Rolling-horizon metrics": RESULTS_DIR / "rolling_horizon_metrics.csv",
    "Counterfactual recommendations": RESULTS_DIR / "counterfactual_recommendations.csv",
    "Counterfactual metrics": RESULTS_DIR / "counterfactual_metrics.csv",
    "Final operational metrics": RESULTS_DIR / "final_operational_metrics.csv",
    "Final strategy summary": RESULTS_DIR / "final_strategy_summary.csv",
    "Final model comparison": RESULTS_DIR / "final_model_comparison.csv",
    "Final sensitivity analysis": RESULTS_DIR / "final_sensitivity_analysis.csv",
    "Final bootstrap intervals": RESULTS_DIR / "final_bootstrap_confidence_intervals.csv",
    "Final renewable absorption figure": FIGURES_DIR / "final_renewable_absorption.png",
    "Final residual surplus figure": FIGURES_DIR / "final_residual_surplus.png",
    "Final grid energy figure": FIGURES_DIR / "final_grid_energy.png",
    "Final service rate figure": FIGURES_DIR / "final_service_rate.png",
    "Final dissertation document": ROOT / "Drafts" / "SEND_EV_Masters_Dissertation_Final.docx",
}

rows = []
for name, output_path in expected_outputs.items():
    exists = output_path.exists()
    rows.append({
        "output": name,
        "path": str(output_path.relative_to(ROOT)),
        "status": "FOUND" if exists else "MISSING",
    })

output_audit = pd.DataFrame(rows)
display(output_audit)

missing = output_audit.loc[output_audit["status"].eq("MISSING"), "output"].tolist()
if missing:
    print("Missing outputs:", ", ".join(missing))
else:
    print("All expected summary outputs are present.")

,output,path,status
0,SEND repaired 2022,Data/DEOP/2022_DEOP_Repaired.csv,FOUND
1,SEND repaired 2023,Data/DEOP/2023_DEOP_Repaired.csv,FOUND
2,Solcast 2022,Data/Solcast/Solcast_2022.csv,FOUND
3,Solcast 2023,Data/Solcast/Solcast_2023.csv,FOUND
4,Surplus time series,Data/Processed/send_solcast_surplus_5min.parquet,MISSING
5,Surplus events,Data/Results/surplus_events.csv,FOUND
6,ACN acquisition manifest,Data/ACN/acquired/acquisition_manifest.csv,FOUND
7,ACN cleaned sessions,Data/ACN/processed/acn_sessions_cleaned.parquet,MISSING
8,ACN optimisation-ready sessions,Data/ACN/processed/acn_sessions_optimisation_ready.parquet,MISSING
9,Keele EV scenarios,Data/ACN/processed/keele_ev_scenarios.parquet,FOUND


Missing outputs: Surplus time series, ACN cleaned sessions, ACN optimisation-ready sessions, Final dissertation document


## Data Sources

SEND/DEOP gives campus consumption, wind generation and PV generation at 5-minute intervals.

Solcast adds weather and irradiance data.

ACN provides workplace EV charging behaviour. It is used to make synthetic Keele scenarios; it is not direct Keele driver data.

The surplus value is a potential surplus proxy, not direct measured curtailment.

In [3]:
send = read_table(PROCESSED_DIR / "send_solcast_surplus_5min.parquet", parse_dates=["DateTime"])
send["DateTime"] = pd.to_datetime(send["DateTime"], errors="coerce")
send = send.dropna(subset=["DateTime"]).sort_values("DateTime")

surplus_events = read_table(RESULTS_DIR / "surplus_events.csv")
daily_surplus = read_table(RESULTS_DIR / "daily_surplus_summary.csv")

send_summary = pd.DataFrame([
    {
        "records": len(send),
        "start": send["DateTime"].min(),
        "end": send["DateTime"].max(),
        "mean_surplus_kw": pd.to_numeric(send["surplus_kw"], errors="coerce").mean(),
        "max_surplus_kw": pd.to_numeric(send["surplus_kw"], errors="coerce").max(),
        "total_surplus_kwh": pd.to_numeric(send["surplus_kw"], errors="coerce").clip(lower=0).sum() * INTERVAL_HOURS,
        "surplus_events": len(surplus_events),
        "surplus_days": daily_surplus["date"].nunique() if "date" in daily_surplus.columns else len(daily_surplus),
    }
])

display(send_summary.round(3))

/var/folders/yd/5b21hsj56kn7zqtz63fs0chh0000gn/T/ipykernel_38689/490156507.py:21: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(send_summary.round(3))


,records,start,end,mean_surplus_kw,max_surplus_kw,total_surplus_kwh,surplus_events,surplus_days
0,193247,2022-03-01,2023-12-31 23:55:00,70.876,3500.059,1070228.396,4035,671


## EV Data Preparation

Notebook 11A standardises the ACN files. Notebook 11B cleans them, checks quality, estimates flexibility and saves EV sessions for optimisation.

The key handoff file is `Data/ACN/processed/acn_sessions_optimisation_ready.parquet`.

In [4]:
manifest = read_table(ACN_DIR / "acquired" / "acquisition_manifest.csv")
count_validation = read_table(RESULTS_DIR / "acn_count_validation.csv")
site_statistics = read_table(RESULTS_DIR / "acn_site_statistics.csv")

manifest_columns = [
    "site", "site_complete", "raw_sessions", "standardised_sessions",
    "arrival_parse_share", "departure_parse_share", "energy_numeric_share",
    "standardisation_action",
]
display(manifest[[column for column in manifest_columns if column in manifest.columns]])

display(count_validation)
display(site_statistics)

,site,site_complete,raw_sessions,standardised_sessions,arrival_parse_share,departure_parse_share,energy_numeric_share,standardisation_action
0,caltech,True,10617,10617,1.0,1.0,1.0,created
1,jpl,True,17411,17411,1.0,1.0,1.0,created
2,office001,True,922,922,1.0,1.0,1.0,created


,site,raw_sessions,cleaned_sessions,optimisation_ready_sessions,removed_by_cleaning,removed_for_optimisation
0,caltech,10617,10594,9867,23,727
1,jpl,17411,17409,16494,2,915
2,office001,922,921,767,1,154


,site,sessions,first_arrival,last_arrival,median_arrival_hour,median_departure_hour,median_energy_delivered_kwh,mean_energy_delivered_kwh,median_energy_required_kwh,median_dwell_hours,median_minimum_charge_hours,median_flexibility_hours,flexible_session_share,overnight_share,assumed_power_share,infeasible_under_assumed_power_share
0,caltech,10594,2019-01-01 03:45:49+00:00,2019-12-31 21:57:23+00:00,10.250000,17.250000,6.233,8.922273,6.233,5.808472,1.049327,3.956296,0.931376,0.047857,1.0,0.068624
1,jpl,17409,2019-01-02 01:00:51+00:00,2019-12-31 22:17:15+00:00,7.383333,16.550000,11.506,14.568820,11.506,8.426389,1.937037,4.690665,0.947441,0.013097,1.0,0.052559
2,office001,921,2019-03-25 16:24:42+00:00,2019-12-31 21:35:43+00:00,11.183333,17.166667,12.337,15.445924,12.337,5.881389,2.076936,2.888324,0.832790,0.001086,1.0,0.167210


## EV Scenario Generation

The cleaned ACN sessions are placed onto Keele surplus-rich days.

Each scenario uses an assumed fleet size, participation rate, charging efficiency and site power limit. This creates a test bed for shifting EV charging into surplus periods.

In [5]:
scenario_summary = read_table(RESULTS_DIR / "scenario_summary.csv", parse_dates=["keele_date"])
scenarios = read_table(ACN_DIR / "processed" / "keele_ev_scenarios.parquet")

scenario_overview = pd.DataFrame([
    {
        "scenario_count": scenario_summary["scenario_id"].nunique(),
        "vehicle_rows": len(scenarios),
        "keele_dates": scenario_summary["keele_date"].nunique(),
        "sites": ", ".join(sorted(scenario_summary["scenario_site"].astype(str).unique())),
        "fleet_sizes": ", ".join(map(str, sorted(scenario_summary["fleet_size"].unique()))),
        "participation_rates": ", ".join(map(str, sorted(scenario_summary["participation_rate"].unique()))),
    }
])

display(scenario_overview)
display(scenario_summary.head(10))

,scenario_count,vehicle_rows,keele_dates,sites,fleet_sizes,participation_rates
0,1280,36960,10,"caltech, jpl, office001, pooled","10, 25, 50, 100","0.25, 0.5, 0.75, 1.0"


,scenario_id,keele_date,scenario_site,fleet_size,participation_rate,participant_count,random_seed,site_limit_kw,total_energy_required_kwh,median_dwell_hours,energy_clipped_sessions
0,20230707_pooled_F10_P025_S42,2023-07-07,pooled,10,0.25,2,42,50.0,14.725,2.966667,0
1,20230707_pooled_F10_P025_S43,2023-07-07,pooled,10,0.25,2,43,50.0,16.763,7.741667,0
2,20230707_pooled_F10_P050_S42,2023-07-07,pooled,10,0.50,5,42,50.0,64.482,3.633333,2
3,20230707_pooled_F10_P050_S43,2023-07-07,pooled,10,0.50,5,43,50.0,39.417,7.533333,2
4,20230707_pooled_F10_P075_S42,2023-07-07,pooled,10,0.75,8,42,50.0,96.707,6.016667,2
5,20230707_pooled_F10_P075_S43,2023-07-07,pooled,10,0.75,8,43,50.0,55.874,7.258333,2
6,20230707_pooled_F10_P100_S42,2023-07-07,pooled,10,1.00,10,42,50.0,106.681,7.133333,2
7,20230707_pooled_F10_P100_S43,2023-07-07,pooled,10,1.00,10,43,50.0,57.448,7.258333,2
8,20230707_pooled_F25_P025_S42,2023-07-07,pooled,25,0.25,6,42,100.0,82.078,4.200000,2
9,20230707_pooled_F25_P025_S43,2023-07-07,pooled,25,0.25,6,43,100.0,49.424,7.200000,2


## Forecasting and Forecast Explanation

The forecasting work predicts the surplus signal used by the charging scheduler. The value of the model is judged by its effect on charging decisions, not only by error scores.

`16_SHAP_Analysis.ipynb` explains the surplus model. If SHAP is not available locally, it uses XGBoost tree contributions instead.

In [6]:
predictions = read_table(RESULTS_DIR / "surplus_predictions.csv", parse_dates=["DateTime"])
importance = read_table(RESULTS_DIR / "shap_feature_importance.csv")

prediction_summary = pd.DataFrame([
    {
        "prediction_rows": len(predictions),
        "start": predictions["DateTime"].min(),
        "end": predictions["DateTime"].max(),
        "mean_actual_surplus_kw": predictions["surplus_actual_kw"].mean(),
        "mean_predicted_surplus_kw": predictions["surplus_pred_xgb"].mean(),
        "backend": importance["explanation_backend"].iloc[0] if "explanation_backend" in importance.columns and len(importance) else "unknown",
    }
])

display(prediction_summary.round(3))
display(importance.head(15))

/var/folders/yd/5b21hsj56kn7zqtz63fs0chh0000gn/T/ipykernel_38689/3574368786.py:15: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(prediction_summary.round(3))


,prediction_rows,start,end,mean_actual_surplus_kw,mean_predicted_surplus_kw,backend
0,35837,2023-08-29 13:35:00,2023-12-31 23:55:00,40.306,35.36,xgboost.pred_contribs


,feature,mean_abs_shap,explanation_backend
0,surplus_kw_lag_1,52.630470,xgboost.pred_contribs
1,ghi,25.797388,xgboost.pred_contribs
2,surplus_kw_rolling_1h,17.766844,xgboost.pred_contribs
3,gti,13.017231,xgboost.pred_contribs
4,solar_kw_lag_1,8.989111,xgboost.pred_contribs
5,solar_kw_rolling_1h,7.214214,xgboost.pred_contribs
6,cloud_opacity,2.940289,xgboost.pred_contribs
7,surplus_kw_rolling_3h,2.760853,xgboost.pred_contribs
8,consumption_kw_rolling_1h,2.598287,xgboost.pred_contribs
9,consumption_kw_lag_1,2.538058,xgboost.pred_contribs


## Charging Baselines, Optimisation and Control

The project compares immediate charging, delayed charging, optimisation and rolling-horizon control.

The default notebooks use small scenario subsets so they run quickly. Set `MAX_SCENARIOS` to `None` when running the full experiment.

In [7]:
operational = read_table(RESULTS_DIR / "final_operational_metrics.csv")
comparison = read_table(RESULTS_DIR / "final_model_comparison.csv")

metric_columns = [
    "strategy", "scenarios", "service_rate", "renewable_absorbed_kwh",
    "residual_surplus_kwh", "ev_grid_energy_kwh", "unserved_energy_kwh", "peak_ev_kw",
]
metric_columns = [column for column in metric_columns if column in operational.columns]

display(operational[metric_columns].sort_values("renewable_absorbed_kwh", ascending=False).reset_index(drop=True))

if {"strategy", "renewable_absorption_change_kwh", "residual_surplus_reduction_kwh", "service_rate_change"}.issubset(comparison.columns):
    improvements = comparison.loc[
        ~comparison["strategy"].eq("immediate"),
        ["scenario_id", "strategy", "forecast_source", "renewable_absorption_change_kwh", "residual_surplus_reduction_kwh", "service_rate_change"],
    ]
    display(improvements.head(12))

,strategy,scenarios,service_rate,renewable_absorbed_kwh,residual_surplus_kwh,ev_grid_energy_kwh,unserved_energy_kwh,peak_ev_kw
0,optimised_actual,24,1.000,584.140,12111.464,32.484,0.000,80.275
1,fixed_delayed,24,0.981,159.460,12536.144,35.096,0.098,73.000
2,immediate,24,1.000,136.035,12559.570,58.703,0.000,38.601
3,rolling_actual,6,1.000,37.377,12603.593,15.877,0.000,15.400


,scenario_id,strategy,forecast_source,renewable_absorption_change_kwh,residual_surplus_reduction_kwh,service_rate_change
1,20230707_pooled_F10_P025_S42,fixed_delayed,none,0.000000,0.000000,0.000000
3,20230707_pooled_F10_P025_S43,fixed_delayed,none,0.000000,0.000000,0.000000
5,20230707_pooled_F10_P050_S42,fixed_delayed,none,7.700000,7.700000,0.000000
7,20230707_pooled_F10_P050_S43,fixed_delayed,none,6.600000,6.600000,0.000000
9,20230707_pooled_F10_P075_S42,fixed_delayed,none,8.537778,8.537778,0.000000
11,20230707_pooled_F10_P075_S43,fixed_delayed,none,12.100000,12.100000,0.000000
13,20230707_pooled_F10_P100_S42,fixed_delayed,none,8.537778,8.537778,0.000000
15,20230707_pooled_F10_P100_S43,fixed_delayed,none,12.525556,12.525556,-0.100000
17,20230707_pooled_F25_P025_S42,fixed_delayed,none,7.700000,7.700000,0.000000
19,20230707_pooled_F25_P025_S43,fixed_delayed,none,6.600000,6.600000,0.000000


## Counterfactual Recommendations

The counterfactual notebook compares baseline charging with the optimised schedule.

It reports practical schedule changes, such as delaying the start time or moving more energy into surplus windows.

In [8]:
counterfactual_metrics = read_table(RESULTS_DIR / "counterfactual_metrics.csv")
counterfactuals = read_table(RESULTS_DIR / "counterfactual_recommendations.csv")

display(counterfactual_metrics)

recommendation_columns = [
    "scenario_id", "strategy", "vehicle_id", "feasible", "changed_intervals",
    "schedule_distance_kwh", "additional_surplus_window_energy_kwh",
]
recommendation_columns = [column for column in recommendation_columns if column in counterfactuals.columns]

display(counterfactuals[recommendation_columns].head(12))

,strategy,explanations,feasible_share,median_changed_intervals,median_schedule_distance_kwh,mean_additional_surplus_window_energy_kwh
0,optimised_actual,424,0.995283,64.0,33.228887,25.364479


,scenario_id,strategy,vehicle_id,changed_intervals,schedule_distance_kwh,additional_surplus_window_energy_kwh
0,20230707_pooled_F10_P025_S42,optimised_actual,EV001,27,1.441889e+01,1.441889e+01
1,20230707_pooled_F10_P025_S42,optimised_actual,EV002,16,8.270000e+00,8.270000e+00
2,20230707_pooled_F10_P025_S43,optimised_actual,EV001,70,3.849222e+01,3.849222e+01
3,20230707_pooled_F10_P025_S43,optimised_actual,EV002,72,3.913222e+01,3.913222e+01
4,20230707_pooled_F10_P050_S42,optimised_actual,EV001,27,1.441889e+01,1.441889e+01
5,20230707_pooled_F10_P050_S42,optimised_actual,EV002,16,8.270000e+00,8.270000e+00
6,20230707_pooled_F10_P050_S42,optimised_actual,EV003,0,1.123990e-12,-1.234568e-13
7,20230707_pooled_F10_P050_S42,optimised_actual,EV004,0,5.001925e-13,-1.865175e-13
8,20230707_pooled_F10_P050_S42,optimised_actual,EV005,109,5.988444e+01,5.218444e+01
9,20230707_pooled_F10_P050_S43,optimised_actual,EV001,70,3.849222e+01,3.849222e+01


## Main Outputs

Important generated artefacts are:

- `Data/Processed/send_solcast_surplus_5min.parquet`
- `Data/ACN/processed/acn_sessions_cleaned.parquet`
- `Data/ACN/processed/acn_sessions_optimisation_ready.parquet`
- `Data/ACN/processed/keele_ev_scenarios.parquet`
- `Data/Results/surplus_predictions.csv`
- `Data/Results/shap_feature_importance.csv`
- `Data/Results/baseline_metrics.csv`
- `Data/Results/optimisation_metrics.csv`
- `Data/Results/rolling_horizon_metrics.csv`
- `Data/Results/counterfactual_recommendations.csv`
- `Data/Results/counterfactual_metrics.csv`
- `Data/Results/final_model_comparison.csv`
- `Data/Results/final_operational_metrics.csv`
- `Data/Results/final_strategy_summary.csv`
- `Data/Results/final_sensitivity_analysis.csv`
- `Data/Results/final_bootstrap_confidence_intervals.csv`
- `Drafts/SEND_EV_Masters_Dissertation_Final.docx`

Important figures are:

- `Figures/renewable_surplus_profile.png`
- `Figures/acn_arrival_distribution.png`
- `Figures/acn_energy_distribution.png`
- `Figures/acn_flexibility_distribution.png`
- `Figures/baseline_charging_profile.png`
- `Figures/optimised_charging_profile.png`
- `Figures/shap_beeswarm.png`
- `Figures/shap_global_bar.png`
- `Figures/shap_waterfall_high_surplus.png`
- `Figures/final_renewable_absorption.png`
- `Figures/final_residual_surplus.png`
- `Figures/final_service_rate.png`
- `Figures/final_grid_energy.png`

## Interpretation Notes and Limitations

- ACN data is used to build synthetic Keele scenarios; it is not observed Keele charging.
- ACN uses 2019 sessions, while SEND/Solcast surplus data uses 2022 and 2023.
- Potential surplus is not the same as measured curtailment.
- Some notebooks use capped scenario counts for quick checking.
- Perfect-information optimisation is a best-case benchmark.
- Forecast and rolling-horizon results are closer to real operation.
- Repaired solar and wind values are estimates, so surplus results inherit those assumptions.
- Vehicle-to-grid, battery degradation, tariff response and user override behaviour are outside the current scope.

## Conclusion

The project moves from energy forecasting into EV demand-response planning.

It builds a renewable-surplus signal, creates synthetic Keele EV scenarios, compares charging strategies and explains how charging could shift to use more renewable energy.

The main message is that forecasted surplus can support explainable EV charging actions, but the operational gain depends on timing accuracy and realistic flexibility assumptions.